# 02 · Run the grid (resumable, two GPUs)

**Settings:** Accelerator **GPU T4 x2**, Internet off, *Save Version → Save & Run All (Commit)*.
**Inputs:** `bbd-code`, the output of notebook 01, and — from the second version on — the output of the
**previous version of this notebook** (Add Input → Your Work → this notebook).

Each committed version runs until its time budget (default 11.3 h, inside Kaggle's 12 h limit), then stops
cleanly. Everything finished is saved under `/kaggle/working/results`. The next version carries those results
forward and continues where the last one stopped; finished runs are never repeated. Order of work:
tuning (8 trials per head and loss) → the two models sent to hardware → seeds 0–4 of the core grid →
references → seeds 5–9 → scale sweep → ablations. You can start the hardware notebook as soon as the first
version has finished (the hardware models run first).

In [ ]:
# --- bootstrap: find the bbd-code dataset, copy it to a writable folder, import it
import glob, os, shutil, sys
# only the bbd-code dataset counts: attached notebook outputs never contain the code (it is copied to /tmp)
hits = [h for h in sorted(glob.glob("/kaggle/input/**/bbd/__init__.py", recursive=True)) if "/bbd_code/" not in h]
hits = hits or sorted(glob.glob("./bbd/__init__.py"))
if not hits:
    raise FileNotFoundError("Attach the 'bbd-code' dataset to this notebook (KAGGLE_INSTRUCTIONS.md, step 2).")
if len(hits) > 1:
    print("warning: several copies of the code are attached; using", hits[0])
SRC = os.path.dirname(os.path.dirname(os.path.abspath(hits[0])))
CODE = "/tmp/bbd_code" if os.path.isdir("/kaggle/working") else SRC
if CODE != SRC:
    shutil.rmtree(CODE, ignore_errors=True)
    shutil.copytree(SRC, CODE, ignore=shutil.ignore_patterns("__pycache__", "*.pyc", "results", "work"))
sys.path.insert(0, CODE)
os.environ["PYTHONPATH"] = CODE + os.pathsep + os.environ.get("PYTHONPATH", "")
import bbd
print("bbd", bbd.__version__, "loaded from", SRC)

In [ ]:
import json, subprocess, time, torch
from bbd import config, runner
BUDGET_HOURS = float(os.environ.get("BBD_BUDGET_HOURS", 11.3))   # stop launching runs; Kaggle stops a session at 12 h
TIERS = os.environ.get("BBD_TIERS", "hw,core,sweep,ablation,ref")
N_WORKERS = max(1, torch.cuda.device_count())
START = time.time(); DEADLINE = START + BUDGET_HOURS * 3600
RESULTS = config.RESULTS_DIR; RESULTS.mkdir(parents=True, exist_ok=True)
n = runner.carry_forward(RESULTS, config.INPUT_ROOT)
print(f"carried forward {n} files; workers: {N_WORKERS}")
from bbd.data import find_prepared_dir
for ds in config.DATASETS:
    print(ds, "prepared at", find_prepared_dir(ds))
print(json.dumps(runner.status(RESULTS), indent=1))

In [ ]:
def launch(stage, deadline=DEADLINE):
    """Start one worker per GPU, stream progress, return when all workers exit."""
    launch_id = f"{stage}-{int(time.time())}"
    (RESULTS / "logs").mkdir(exist_ok=True)
    procs = []
    for i in range(N_WORKERS):
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(i) if torch.cuda.is_available() else "",
                   BBD_LAUNCH=launch_id, BBD_THREADS="2")
        log = open(RESULTS / "logs" / f"{launch_id}_w{i}.log", "w")
        cmd = [sys.executable, "-m", "bbd.runner", stage, "--worker", str(i), "--n-workers", str(N_WORKERS),
               "--deadline", str(deadline), "--tiers", TIERS, "--results", str(RESULTS)]
        procs.append((subprocess.Popen(cmd, env=env, stdout=log, stderr=subprocess.STDOUT, cwd=CODE), log))
    last = 0
    while any(p.poll() is None for p, _ in procs):
        time.sleep(30)
        if time.time() - last > 600:
            last = time.time()
            print(time.strftime("%H:%M"), json.dumps(runner.status(RESULTS)), flush=True)
            for i in range(N_WORKERS):
                tail = (RESULTS / "logs" / f"{launch_id}_w{i}.log").read_text().splitlines()[-2:]
                print(f"   w{i}:", *tail, sep="\n      ")
    for p, log in procs:
        log.close()
    return [p.returncode for p, _ in procs]

## Tuning (skipped once complete)

In [ ]:
st = runner.status(RESULTS)
if st["tuning"][0] < st["tuning"][1]:
    print("tuning exit codes:", launch("tune"))
hps = runner.best_hyperparameters(RESULTS)
print(len(hps), "tuned (dataset | family | loss) keys")
import pandas as pd
display(pd.DataFrame([dict(key=k, **v) for k, v in sorted(hps.items())]))

## Grid

In [ ]:
if time.time() < DEADLINE - 600:
    print("grid exit codes:", launch("run"))
print(json.dumps(runner.status(RESULTS), indent=1))

In [ ]:
errs = [r for r in runner.read_records(RESULTS) if r.get("stage") == "error"]
print(len(errs), "errors")
for r in errs[-5:]:
    print(r["run_id"], r["error"]); print(r["traceback"][-600:])

## Quick look (sanity check, not the analysis)

In [ ]:
from bbd import stats
df = stats.records_frame(RESULTS)
if len(df):
    q = df[(df.variant.isin(["raw", "posthoc1"])) & (df.tags_str == "")]
    piv = q.pivot_table(index=["dataset", "model", "s_label"], columns=["rho_str", "cond"], values="macro_f1", aggfunc="mean")
    display((100 * piv).round(1))